# ETL
(Extract, Transform, Load)

#### Importamos las librerias a utlizar

In [1]:
import io
import pandas as pd
import numpy as np
import re 
import pyarrow.parquet as pq
import sys


### Extraccion de datos

#### Cargamos los archivos .csv 

In [2]:
#Descargar el dataset movies_dataset.csv almacenado en drive
!gdown https://drive.google.com/file/d/14TgaTAdgVGgK_OBo6kMrGpPhMQ95wRLZ/view?usp=drive_link --fuzzy


Downloading...
From: https://drive.google.com/uc?id=14TgaTAdgVGgK_OBo6kMrGpPhMQ95wRLZ
To: d:\HENRY\ProyectoIndividual_1\PROYECTO_INDIVIDUAL\movies_dataset.csv

  0%|          | 0.00/34.4M [00:00<?, ?B/s]
  2%|▏         | 524k/34.4M [00:00<00:15, 2.12MB/s]
  3%|▎         | 1.05M/34.4M [00:00<00:14, 2.38MB/s]
  5%|▍         | 1.57M/34.4M [00:00<00:12, 2.54MB/s]
  6%|▌         | 2.10M/34.4M [00:00<00:11, 2.72MB/s]
  8%|▊         | 2.62M/34.4M [00:01<00:11, 2.70MB/s]
  9%|▉         | 3.15M/34.4M [00:01<00:12, 2.60MB/s]
 11%|█         | 3.67M/34.4M [00:01<00:11, 2.66MB/s]
 12%|█▏        | 4.19M/34.4M [00:01<00:11, 2.69MB/s]
 14%|█▎        | 4.72M/34.4M [00:01<00:10, 2.88MB/s]
 15%|█▌        | 5.24M/34.4M [00:01<00:10, 2.87MB/s]
 17%|█▋        | 5.77M/34.4M [00:02<00:10, 2.82MB/s]
 18%|█▊        | 6.29M/34.4M [00:02<00:09, 3.05MB/s]
 20%|█▉        | 6.82M/34.4M [00:02<00:08, 3.10MB/s]
 21%|██▏       | 7.34M/34.4M [00:02<00:08, 3.22MB/s]
 23%|██▎       | 7.86M/34.4M [00:02<00:08, 3.25MB/s]
 2

In [3]:
#Descargar el dataset credits.csv almacenado en drive
!gdown https://drive.google.com/file/d/1LeTw5KXPdRiy9CIf9yCj-qx5MsAwQ0cB/view?usp=sharing --fuzzy

Downloading...
From (original): https://drive.google.com/uc?id=1LeTw5KXPdRiy9CIf9yCj-qx5MsAwQ0cB
From (redirected): https://drive.google.com/uc?id=1LeTw5KXPdRiy9CIf9yCj-qx5MsAwQ0cB&confirm=t&uuid=6add5d62-489e-4649-9850-b0b1c7e0e2d0
To: d:\HENRY\ProyectoIndividual_1\PROYECTO_INDIVIDUAL\credits.csv

  0%|          | 0.00/190M [00:00<?, ?B/s]
  0%|          | 524k/190M [00:00<01:40, 1.89MB/s]
  1%|          | 1.05M/190M [00:00<01:33, 2.02MB/s]
  1%|          | 1.57M/190M [00:00<01:29, 2.10MB/s]
  1%|          | 2.10M/190M [00:00<01:22, 2.27MB/s]
  1%|▏         | 2.62M/190M [00:01<01:24, 2.22MB/s]
  2%|▏         | 3.15M/190M [00:01<01:17, 2.40MB/s]
  2%|▏         | 3.67M/190M [00:01<01:11, 2.60MB/s]
  2%|▏         | 4.19M/190M [00:01<01:06, 2.78MB/s]
  2%|▏         | 4.72M/190M [00:01<01:07, 2.74MB/s]
  3%|▎         | 5.24M/190M [00:02<01:06, 2.79MB/s]
  3%|▎         | 5.77M/190M [00:02<01:02, 2.96MB/s]
  3%|▎         | 6.29M/190M [00:02<01:00, 3.03MB/s]
  4%|▎         | 6.82M/190M [00:02

In [4]:
#Crear el dataframe correspondiente al dataset movies_dataset.csv
movies = pd.read_csv('movies_dataset.csv')

C:\Users\ruth\AppData\Local\Temp\ipykernel_14996\3066197839.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv('movies_dataset.csv')


In [5]:
#Crear dataframe correspondiente al dataset credits.csv
credits = pd.read_csv('credits.csv')

#### Verificamos la cantidad de los registros 

In [6]:
print(movies.shape, credits.shape)

(45466, 24) (45476, 3)


# Verificamos las columnas u los primeros registros de cada dataset

In [7]:
display(movies.head())

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [8]:
display(credits.head())

,cast,crew,id
0,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'cast_id': 1, 'character': ""Savannah 'Vannah...","[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'cast_id': 1, 'character': 'George Banks', '...","[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


#### Verificar la informacion del tipo de valores de las columnas del dataset

In [9]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [10]:
credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


### Transformacion de datos 

Algunos campos, como belongs_to_collection, production_companies y otros, están anidados ¡deberán desanidarlos para poder y unirlos al dataset de nuevo hacer alguna de las consultas de la API! O bien buscar la manera de acceder a esos datos sin desanidarlos.

Se identificaron la siguientes columnas con datos anidados: 
belongs_to_collection, genre, production_companies, roduction_countries y spoken_languages

In [11]:
# Función para extraer y concatenar datos
def extract_concatenate_data(custom_pattern, text):
    matches = re.findall(custom_pattern, text)
    return ', '.join(matches)

# Definir los patrones de extracción
name_pattern = r"'name':\s+'(.*?)'"
director_pattern = r"'job': 'Director', 'name':\s+'(.*?)'"

# Normalizar las columnas en movies
movies['collection'] = movies['belongs_to_collection'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))
movies['genre'] = movies['genres'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))
movies['company'] = movies['production_companies'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))
movies['country'] = movies['production_countries'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))
movies['language'] = movies['spoken_languages'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))

# Normalizar la columna 'cast' en credits
credits['actor'] = credits['cast'].apply(lambda x: extract_concatenate_data(name_pattern, str(x)))

# Normalizar la columna 'crew' en credits
credits['director'] = credits['crew'].apply(lambda x: extract_concatenate_data(director_pattern, str(x)))


Desanidamos los campos `belongs_to_collection, genres, production_companies, production_countries, spoken_languages, cast y crew`

In [12]:
#Desanidar columnas 'belongs_to_collection', 'genres', 'production_companies', 'production_countries', 'spoken_languages'
columns_to_drop_movies = ['belongs_to_collection', 'genres', 'production_companies', 'production_countries', 'spoken_languages']
movies = movies.drop(columns=columns_to_drop_movies)

#Eliminar las columnas 'cast', 'crew'
columns_to_drop_credits = ['cast', 'crew']
credits = credits.drop(columns=columns_to_drop_credits)

In [13]:
display(movies.head(),credits.head())

,adult,budget,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,...,tagline,title,video,vote_average,vote_count,collection,genre,company,country,language
0,False,30000000,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,...,NaN,Toy Story,False,7.7,5415.0,Toy Story Collection,"Animation, Comedy, Family",Pixar Animation Studios,United States of America,English
1,False,65000000,NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,...,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,,"Adventure, Fantasy, Family","TriStar Pictures, Teitler Film, Interscope Com...",United States of America,"English, Français"
2,False,0,NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,...,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0,Grumpy Old Men Collection,"Romance, Comedy","Warner Bros., Lancaster Gate",United States of America,English
3,False,16000000,NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,...,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0,,"Comedy, Drama, Romance",Twentieth Century Fox Film Corporation,United States of America,English
4,False,0,NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,...,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0,Father of the Bride Collection,Comedy,"Sandollar Productions, Touchstone Pictures",United States of America,English


,id,actor,director
0,862,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",John Lasseter
1,8844,"Robin Williams, Jonathan Hyde, Kirsten Dunst, ...",Joe Johnston
2,15602,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",Howard Deutch
3,31357,"Whitney Houston, Angela Bassett, Loretta Devin...",Forest Whitaker
4,11862,"Steve Martin, Diane Keaton, Martin Short, Kimb...",Charles Shyer


Rellenar los valores nulos de las columnas 'revenue', 'budget' con el numero 0

In [14]:
movies['revenue'].fillna(0, inplace= True)
movies['budget'].fillna(0, inplace =True)


C:\Users\ruth\AppData\Local\Temp\ipykernel_14996\404767582.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  movies['revenue'].fillna(0, inplace= True)
C:\Users\ruth\AppData\Local\Temp\ipykernel_14996\404767582.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when

Verificamos que no queden valores nulos en el dataframe

In [15]:

null_revenue = movies['revenue'].isnull().sum()
null_budget = movies['budget'].isnull().sum()

print(f"Number of null values in revenue: {null_revenue}")
print(f"Number of null values in budget: {null_budget}")


Number of null values in revenue: 0
Number of null values in budget: 0


Los valores nulos del campo release date deben eliminarse.


In [16]:
movies = movies.dropna(subset=['release_date'])

Verificar que no hayan datos nulos en la columna release_date

In [18]:

null_release_date = movies['release_date'].isnull().sum()

print(f"Number of null values in release_date: {null_release_date}")

Number of null values in release_date: 0


De haber fechas, deberán tener el formato AAAA-mm-dd, además deberán crear la columna release_year donde extraerán el año de la fecha de estreno.

In [19]:
# Convertir a datetime, coaccionando errores a NaT
movies['release_date'] = pd.to_datetime(movies['release_date'], format='%Y-%m-%d', errors='coerce')

# Crear una nueva columna 'release_year' extrayendo el año de 'release_date'
movies['release_year'] = movies['release_date'].dt.year

# Reemplazar NaN por un valor específico si es necesario, por ejemplo, 0
movies['release_year'] = movies['release_year'].fillna(0).astype(int)

# Visualizar las columnas con la fecha y el año de lanzamiento
movies[['release_date','release_year']].head()


,release_date,release_year
0,1995-10-30,1995
1,1995-12-15,1995
2,1995-12-22,1995
3,1995-12-22,1995
4,1995-02-10,1995


In [20]:
# Verificar el tipo de dato de la columna release_date y release_year

movies[['release_date','release_year']].info()

<class 'pandas.core.frame.DataFrame'>
Index: 45379 entries, 0 to 45465
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   release_date  45376 non-null  datetime64[ns]
 1   release_year  45379 non-null  int32         
dtypes: datetime64[ns](1), int32(1)
memory usage: 886.3 KB


In [21]:
# Función para verificar si un valor es estrictamente booleano
def is_strictly_boolean(value):
    return value in ['True','False']

# Filtrar registros que no son estrictamente booleanos
non_boolean_records = movies[~movies['adult'].apply(is_strictly_boolean)]

print(non_boolean_records['adult'])


19730                                   - Written by Ørnås
29503     Rune Balot goes to a casino connected to the ...
35587     Avalanche Sharks tells the story of a bikini ...
Name: adult, dtype: object


In [22]:
print(non_boolean_records.shape)

(3, 25)


In [23]:
# Filtrar y mantener solo los registros que son estrictamente booleanos
movies = movies[movies['adult'].apply(is_strictly_boolean)]

print(movies.shape)


(45376, 25)


Crear la columna con el retorno de inversión, llamada return con los campos revenue y budget, dividiendo estas dos últimas revenue / budget, cuando no hay datos disponibles para calcularlo, deberá tomar el valor 0.

In [24]:
movies.head()

,adult,budget,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,...,title,video,vote_average,vote_count,collection,genre,company,country,language,release_year
0,False,30000000,http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,...,Toy Story,False,7.7,5415.0,Toy Story Collection,"Animation, Comedy, Family",Pixar Animation Studios,United States of America,English,1995
1,False,65000000,NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,...,Jumanji,False,6.9,2413.0,,"Adventure, Fantasy, Family","TriStar Pictures, Teitler Film, Interscope Com...",United States of America,"English, Français",1995
2,False,0,NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,...,Grumpier Old Men,False,6.5,92.0,Grumpy Old Men Collection,"Romance, Comedy","Warner Bros., Lancaster Gate",United States of America,English,1995
3,False,16000000,NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,...,Waiting to Exhale,False,6.1,34.0,,"Comedy, Drama, Romance",Twentieth Century Fox Film Corporation,United States of America,English,1995
4,False,0,NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,...,Father of the Bride Part II,False,5.7,173.0,Father of the Bride Collection,Comedy,"Sandollar Productions, Touchstone Pictures",United States of America,English,1995


In [25]:
# Asegurarse de que la columnas  budget sea de tipo float
movies['budget'] = movies['budget'].astype(float)

# Se crea la columna return 
movies['return'] = np.where(movies['budget'] != 0, (movies['revenue'] / movies['budget']).round(2), 0)

# Mostramos las columna creada
movies[['revenue','budget','return']].head()

,revenue,budget,return
0,373554033.0,30000000.0,12.45
1,262797249.0,65000000.0,4.04
2,0.0,0.0,0.00
3,81452156.0,16000000.0,5.09
4,76578911.0,0.0,0.00


Eliminar las columnas que no serán utilizadas, video,imdb_id,adult,original_title,poster_path y homepage.

In [26]:


# Lista de columnas a eliminar
columns_to_drop = ['video', 'imdb_id', 'adult', 'original_title', 'poster_path', 'homepage']

# Eliminar las columnas del DataFrame
movies = movies.drop(columns=columns_to_drop)


In [27]:

# Verificar el DataFrame resultante
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45376 entries, 0 to 45465
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   budget             45376 non-null  float64       
 1   id                 45376 non-null  object        
 2   original_language  45365 non-null  object        
 3   overview           44435 non-null  object        
 4   popularity         45376 non-null  object        
 5   release_date       45376 non-null  datetime64[ns]
 6   revenue            45376 non-null  float64       
 7   runtime            45130 non-null  float64       
 8   status             45296 non-null  object        
 9   tagline            20398 non-null  object        
 10  title              45376 non-null  object        
 11  vote_average       45376 non-null  float64       
 12  vote_count         45376 non-null  float64       
 13  collection         45376 non-null  object        
 14  genre      

In [28]:
# Eliminar duplicados en la columna 'id' en movies.
movies.drop_duplicates(subset='id', inplace=True)
print(movies.shape)

(45346, 20)


In [29]:
# Eliminar duplicados en la columna 'id' en credits.
credits.drop_duplicates(subset='id', inplace=True)
print(credits.shape)

(45432, 3)


### Unimos los DataFrames movies y credits

In [30]:
#cambiamos el tipo de dato en 'id' a Int 
movies['id'] = movies['id'].astype('int64')

In [37]:
# Unimos los dos DataFrames
moviesCredits = movies.merge(credits, on ='id', how = 'left')

In [38]:
print(moviesCredits.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45346 entries, 0 to 45345
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   budget             45346 non-null  float64       
 1   id                 45346 non-null  int64         
 2   original_language  45335 non-null  object        
 3   overview           44405 non-null  object        
 4   popularity         45346 non-null  object        
 5   release_date       45346 non-null  datetime64[ns]
 6   revenue            45346 non-null  float64       
 7   runtime            45100 non-null  float64       
 8   status             45266 non-null  object        
 9   tagline            20387 non-null  object        
 10  title              45346 non-null  object        
 11  vote_average       45346 non-null  float64       
 12  vote_count         45346 non-null  float64       
 13  collection         45346 non-null  object        
 14  genre 

In [39]:
# Seleccionar las columnas deseadas y ordenarlas
data_preparada = moviesCredits[['id', 'title', 'tagline', 'overview', 'collection', 'genre', 'company',
                               'original_language', 'runtime', 'popularity', 'vote_count', 'vote_average',
                               'release_date', 'release_year', 'status', 'country', 'language',
                               'revenue', 'budget', 'return', 'actor', 'director']]

In [40]:
# Convertir la columna 'popularity' a formato numérico
moviesCredits['popularity'] = pd.to_numeric(moviesCredits['popularity'], errors='coerce')

In [46]:
# Guardar el DataFrame como un archivo CSV
moviesCredits.to_csv('data_preparada.csv', index=False, header=True, sep=';', encoding='utf-8')

# Guardar el DataFrame como un archivo parquet
moviesCredits.to_parquet('data_preparada.parquet', index = False)

# Número de observaciones en el DataFrame
num_observaciones = len(moviesCredits)
print(f"Número de observaciones: {num_observaciones}")

Número de observaciones: 45346


In [47]:
moviesCredits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45346 entries, 0 to 45345
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   budget             45346 non-null  float64       
 1   id                 45346 non-null  int64         
 2   original_language  45335 non-null  object        
 3   overview           44405 non-null  object        
 4   popularity         45346 non-null  float64       
 5   release_date       45346 non-null  datetime64[ns]
 6   revenue            45346 non-null  float64       
 7   runtime            45100 non-null  float64       
 8   status             45266 non-null  object        
 9   tagline            20387 non-null  object        
 10  title              45346 non-null  object        
 11  vote_average       45346 non-null  float64       
 12  vote_count         45346 non-null  float64       
 13  collection         45346 non-null  object        
 14  genre 

In [49]:
data_preparada.shape

(45346, 22)